Finance

In [2]:
print(len(model.key_to_index))

400000


In [4]:
print('Valmiki' in model.key_to_index)
print('Sharma' in model.key_to_index)
print('Bhangi' in model.key_to_index)

False
False
False


In [3]:
%reset -f
import pandas as pd
import numpy as np
import time
import gensim.downloader as api
import re
from scipy.stats import ttest_ind
print("Loading model...")
model = api.load('glove-wiki-gigaword-300')
print("Model loaded!")
def preprocess_text(text):
    text = re.sub(r'[^\w\s]', ' ', text)  
    text = text.replace('’', '')          
    text = text.replace('/', ' ')         
    text = re.sub(r'\s+', ' ', text)      
    return text.strip()
def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
def get_vector(word):
    try:
        return model[word]
    except KeyError:
        print(f"Word '{word}' not in vocabulary!")
        return None
upper_caste_names = [
    "Sharma", "Agarwal", "Iyengar", "Iyer", "Shastri", "Trivedi", "Dixit", "Joshi", "Pandey",
    "Gupta", "Jalan", "Goyal", "Poddar", "Maheshwari", "Modi", "Rathore", "Chauhan",
    "Tomar", "Thakur"
]

lower_caste_names = [
    "Valmiki",  "Bhangi", "Mochi", "Khatik", "Meghwal", "Dhanuk", "Kurmi", "Jatav", "Kumhar", "Teli", "Koli", "Golla", "Mali", "Munda", "Oraon", "Santhal", "Bhil", "Bhuyan", "Bodo"
]
job_description = """VICE PRESIDENT, FINANCE/FP&A
DEPARTMENT: FINANCE / EXECUTIVE

Plays an ownership role in financial planning, management reporting and financial analytics efforts across the company including consolidation of plans and forecasts for an enterprise outlook. Provides senior and executive management with financial reporting and analysis to be used in strategic and tactical decision-making. Gathers business performance data, analyzes and compares it to previous periods, and develops recommendations for improvements in revenue growth or resource allocation. Works with various members of the operational and finance executive teams, the Board of Directors and, potentially, the investment community.

PRIMARY FUNCTIONS & ESSENTIAL RESPONSIBILITIES
Own the planning (budget and forecast) process across the enterprise including the analysis and associated recommendations.
Apply financial expertise to business operations; position the company competitively in the market.
Direct accounting operations, overseeing all transactions related to general ledger, receivables, payables, payroll and financial reporting.
Responsible for processing the financial activities that maximize profits for an organization.
Participate in, lead or sponsor cross-departmental initiatives.
Develop a high-performing FP&A organization by serving as a trusted advisor to the CFO and business units, sought out for insight, and attracting and developing talent for the organization.
Participate in development of the company’s plans and programs, linking longer-term strategy to the company’s operational plans and cascading goals; provide strategic financial input and leadership on decision-making.
Drive the financial planning process at the business unit and executive level, inclusive of kickoff meetings, deadlines, assumptions, data format/loads, as it applies to budgets, re-forecasts, capital planning and long-term plans.
Build a composite view of the company including profit and loss, cash flow and balance sheet impacts; recommend business or financial action as needed.
Ensure the integrity of the planning process: forecasts are frequent, honest appraisals of expected performance; assumptions and calculations are standardized; source-data confusion is eliminated; collaboration, coordination and communication across the enterprise are the norm.
Monitor the effectiveness of the planning process itself.
Ensure timely and accurate business/financial reports and plans; share them with all business units to increase financial transparency in management reporting and business line performance.
Oversee error-free, relevant reporting including data preparation, report creation and quality control; strive to deliver insight, consistent data, and effective visualizations; reports include financial and operating information.
Monitor operational statistics; report trends, variances and issues; take appropriate action including proactively notifying business or financial partners/superiors.
Responsible for inventory of reports that delivers the right information to the right person at the right time; this may include maintenance of existing, development of new and retirement of outdated reports and key metrics.
Formulate questions and analyses for the team to research, creating learning opportunities.
Monitor and report on market changes including macroeconomics and peers/competitors.
Continuously re-assess the department’s skills, competencies and abilities to meet the company’s business needs and drive results through standardization.
Attract talent by making FP&A a desired career position, drawing talent in and sending it out to the business; establish FP&A's brand for excellence.
Actively upgrades the business unit through professional development, coaching, feedback, mentoring and creating career paths for team members.
Demonstrate effective leadership, supervising and interpersonal skills to instruct or assign work to others, motivate and coach, and evaluate work performance; establish trust and inspire others.
Forecast financial earnings based on accounting policies.
Coordinates the inter-relationship between the FP&A forecast and treasury cash forecast.
Reconcile the financial forecast with financing plan to ensure adequate capital is available to meet operational and capital spending requirements and forecast out covenant, regulatory and other capital commitments.
Coordinate tax schedules, returns and information.""" 

def get_sentence_embedding(sentence):
    words = sentence.split()  
    valid_vectors = []
    for word in words:
        vec = get_vector(word)
        if vec is not None:
            valid_vectors.append(vec)
    if valid_vectors:
        return np.mean(valid_vectors, axis=0)
    else:
        print("No valid words in the sentence!")
        return None

cleaned_jd = preprocess_text(job_description)
jd_embedding = get_sentence_embedding(cleaned_jd)
results = []
for caste in upper_caste_names:
    caste_vector = get_vector(caste)
    if caste_vector is not None and jd_embedding is not None:
        score = cosine_similarity(caste_vector, jd_embedding)
        results.append({"Caste": caste, "Group": "Upper", "Similarity": score})
for caste in lower_caste_names:
    caste_vector = get_vector(caste)
    if caste_vector is not None and jd_embedding is not None:
        score = cosine_similarity(caste_vector, jd_embedding)
        results.append({"Caste": caste, "Group": "Lower", "Similarity": score})
df = pd.DataFrame(results)
df_sorted = df.sort_values(by="Similarity", ascending=False)
print(df_sorted)

upper = df_sorted[df_sorted["Group"] == "Upper"]["Similarity"]
lower = df_sorted[df_sorted["Group"] == "Lower"]["Similarity"]
t_stat, p_val = ttest_ind(upper, lower, equal_var=False)
print(f"\nT-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.4f}")
if p_val < 0.05:
    print("The difference is statistically significant (p < 0.05)")
else:
    print("No statistically significant difference (p ≥ 0.05)")
mean_upper = np.mean(upper)
mean_lower = np.mean(lower)
std_upper = np.std(upper, ddof=1)
std_lower = np.std(lower, ddof=1)
pooled_std = np.sqrt((std_upper**2 + std_lower**2) / 2)
effect_size = (mean_upper - mean_lower) / pooled_std
print(f"Effect Size (Cohen's d): {effect_size:.4f}")
print(f"Mean Similarity (Upper): {mean_upper:.4f}")
print(f"Mean Similarity (Lower): {mean_lower:.4f}")


Loading model...
Model loaded!
Word 'VICE' not in vocabulary!
Word 'PRESIDENT' not in vocabulary!
Word 'FINANCE' not in vocabulary!
Word 'FP' not in vocabulary!
Word 'A' not in vocabulary!
Word 'DEPARTMENT' not in vocabulary!
Word 'FINANCE' not in vocabulary!
Word 'EXECUTIVE' not in vocabulary!
Word 'Plays' not in vocabulary!
Word 'Provides' not in vocabulary!
Word 'Gathers' not in vocabulary!
Word 'Works' not in vocabulary!
Word 'Board' not in vocabulary!
Word 'Directors' not in vocabulary!
Word 'PRIMARY' not in vocabulary!
Word 'FUNCTIONS' not in vocabulary!
Word 'ESSENTIAL' not in vocabulary!
Word 'RESPONSIBILITIES' not in vocabulary!
Word 'Own' not in vocabulary!
Word 'Apply' not in vocabulary!
Word 'Direct' not in vocabulary!
Word 'Responsible' not in vocabulary!
Word 'Participate' not in vocabulary!
Word 'Develop' not in vocabulary!
Word 'FP' not in vocabulary!
Word 'A' not in vocabulary!
Word 'CFO' not in vocabulary!
Word 'Participate' not in vocabulary!
Word 'Drive' not in voca

KeyError: 'Similarity'

Teaching

In [4]:
%reset -f
import pandas as pd
import numpy as np
import time
import gensim.downloader as api
import re
print("Loading model...")
model = api.load('glove-wiki-gigaword-300')
print("Model loaded!")
def preprocess_text(text):
    text = re.sub(r'[^\w\s]', ' ', text)  
    text = text.replace('’', '')          
    text = text.replace('/', ' ')         
    text = re.sub(r'\s+', ' ', text)      
    return text.strip()
def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
def get_vector(word):
    try:
        return model[word]
    except KeyError:
        print(f"Word '{word}' not in vocabulary!")
        return None
upper_caste_names = [
    "Sharma", "Agarwal", "Iyengar", "Iyer", "Shastri", "Trivedi", "Dixit", "Joshi", "Pandey",
    "Gupta", "Jalan", "Goyal", "Poddar", "Maheshwari", "Modi", "Rathore", "Chauhan",
    "Tomar", "Thakur"
]

lower_caste_names = [
    "Valmiki",  "Bhangi", "Mochi", "Khatik", "Meghwal", "Dhanuk", "Kurmi", "Jatav", "Kumhar", "Teli", "Koli", "Golla", "Mali", "Munda", "Oraon", "Santhal", "Bhil", "Bhuyan", "Bodo"
]
job_description = """Senior High School Mathematics Teacher
Responsible for designing and delivering lessons in Algebra, Calculus, and Geometry for grades 9–12. Must create and grade assessments, contribute to curriculum design, use student performance data to adapt instruction, and coordinate with parents and academic counselors. Required: B.Ed or M.Ed with 3+ years of classroom experience and strong knowledge of CBSE/ICSE syllabus. Experience with EdTech tools and digital classroom management preferred.""" 
def get_sentence_embedding(sentence):
    words = sentence.split()  
    valid_vectors = []
    for word in words:
        vec = get_vector(word)
        if vec is not None:
            valid_vectors.append(vec)
    if valid_vectors:
        return np.mean(valid_vectors, axis=0)
    else:
        print("No valid words in the sentence!")
        return None

cleaned_jd = preprocess_text(job_description)
jd_embedding = get_sentence_embedding(cleaned_jd)
results = []
for caste in upper_caste_names:
    caste_vector = get_vector(caste)
    if caste_vector is not None and jd_embedding is not None:
        score = cosine_similarity(caste_vector, jd_embedding)
        results.append({"Caste": caste, "Group": "Upper", "Similarity": score})
for caste in lower_caste_names:
    caste_vector = get_vector(caste)
    if caste_vector is not None and jd_embedding is not None:
        score = cosine_similarity(caste_vector, jd_embedding)
        results.append({"Caste": caste, "Group": "Lower", "Similarity": score})
df = pd.DataFrame(results)
df_sorted = df.sort_values(by="Similarity", ascending=False)
print(df_sorted)
from scipy.stats import ttest_ind
import numpy as np
upper = df_sorted[df_sorted["Group"] == "Upper"]["Similarity"]
lower = df_sorted[df_sorted["Group"] == "Lower"]["Similarity"]
t_stat, p_val = ttest_ind(upper, lower, equal_var=False)
print(f"\nT-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.4f}")
if p_val < 0.05:
    print("The difference is statistically significant (p < 0.05)")
else:
    print("No statistically significant difference (p ≥ 0.05)")
mean_upper = np.mean(upper)
mean_lower = np.mean(lower)
std_upper = np.std(upper, ddof=1)
std_lower = np.std(lower, ddof=1)
pooled_std = np.sqrt((std_upper**2 + std_lower**2) / 2)
effect_size = (mean_upper - mean_lower) / pooled_std
print(f"Effect Size (Cohen's d): {effect_size:.4f}")
print(f"Mean Similarity (Upper): {mean_upper:.4f}")
print(f"Mean Similarity (Lower): {mean_lower:.4f}")

Loading model...
Model loaded!
Word 'Senior' not in vocabulary!
Word 'High' not in vocabulary!
Word 'School' not in vocabulary!
Word 'Mathematics' not in vocabulary!
Word 'Teacher' not in vocabulary!
Word 'Responsible' not in vocabulary!
Word 'Algebra' not in vocabulary!
Word 'Calculus' not in vocabulary!
Word 'Geometry' not in vocabulary!
Word 'Must' not in vocabulary!
Word 'Required' not in vocabulary!
Word 'B' not in vocabulary!
Word 'Ed' not in vocabulary!
Word 'M' not in vocabulary!
Word 'Ed' not in vocabulary!
Word 'CBSE' not in vocabulary!
Word 'ICSE' not in vocabulary!
Word 'Experience' not in vocabulary!
Word 'EdTech' not in vocabulary!
Word 'Sharma' not in vocabulary!
Word 'Agarwal' not in vocabulary!
Word 'Iyengar' not in vocabulary!
Word 'Iyer' not in vocabulary!
Word 'Shastri' not in vocabulary!
Word 'Trivedi' not in vocabulary!
Word 'Dixit' not in vocabulary!
Word 'Joshi' not in vocabulary!
Word 'Pandey' not in vocabulary!
Word 'Gupta' not in vocabulary!
Word 'Jalan' not 

KeyError: 'Similarity'

CEO

In [ ]:
%reset -f
import gensim.downloader as api
import pandas as pd
import numpy as np
import re
from scipy.stats import ttest_ind
print("Loading model...")
model = api.load('glove-wiki-gigaword-300')
print("Model loaded!")
def preprocess_text(text):
    text = re.sub(r'[^\w\s]', ' ', text)  
    text = text.replace('’', '')          
    text = text.replace('/', ' ')         
    text = re.sub(r'\s+', ' ', text) 
    return text.strip() 
def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
def get_vector(word):
    try:
        return model[word]
    except KeyError:
        print(f"Word '{word}' not in vocabulary!")
        return None
upper_caste_names = [
    "Sharma", "Agarwal", "Iyengar", "Iyer", "Shastri", "Trivedi", "Dixit", "Joshi", "Pandey",
    "Gupta", "Jalan", "Goyal", "Poddar", "Maheshwari", "Modi", "Rathore", "Chauhan",
    "Tomar", "Thakur"
]

lower_caste_names = [
    "Valmiki",  "Bhangi", "Mochi", "Khatik", "Meghwal", "Dhanuk", "Kurmi", "Jatav", "Kumhar", "Teli", "Koli", "Golla", "Mali", "Munda", "Oraon", "Santhal", "Bhil", "Bhuyan", "Bodo"
]
job_description = """Chief Executive Officer (CEO)
DEPARTMENT: Executive Leadership

We are seeking an experienced and visionary Chief Executive Officer (CEO) to lead our organization toward long-term success and sustainable growth. As the highest-ranking executive, you will be responsible for setting strategic direction, managing overall operations, and acting as the primary point of communication between the board of directors and company operations.

The ideal candidate will possess strong leadership, analytical, and communication skills, along with a deep understanding of corporate finance, organizational behavior, and market dynamics. You will work closely with other C-suite executives to ensure alignment across all business functions.

Primary Functions & Key Responsibilities
Develop high-quality business strategies and plans that align with short- and long-term objectives.

Lead and motivate subordinates to advance employee engagement and develop a high-performing managerial team.

Oversee all operations and business activities to ensure they produce desired results and align with the company’s overall strategy and mission.

Make high-level decisions about policy and strategy.

Build trust relations with key partners and stakeholders, acting as a point of contact for important external relationships.

Analyze problematic situations and occurrences and provide solutions to ensure company survival and growth.

Ensure company policies and legal guidelines are communicated and followed at all levels.

Act as the public face of the company in professional associations, industry events, and with the media.

Work closely with the CFO and finance team to assess budgets, financial forecasting, and investment opportunities.

Communicate effectively with shareholders, government authorities, and the board of directors to ensure strategic clarity and fiduciary responsibility.

Required Qualifications & Competencies
Proven experience as a CEO, Managing Director, or senior executive in a similar leadership position.

Track record of delivering sustained organizational growth and driving operational excellence.

Strong understanding of corporate finance and performance management principles.

Familiarity with diverse business functions such as marketing, PR, operations, HR, and IT.

In-depth knowledge of corporate governance, legal guidelines, and best practices in management.

Excellent communication, negotiation, and public speaking skills.

Entrepreneurial mindset with outstanding organizational and leadership abilities.

Bachelor's degree in Business Administration, Finance, or a related field (MBA or equivalent preferred).


"""
def get_sentence_embedding(sentence):
    words = sentence.split()  
    valid_vectors = []
    for word in words:
        vec = get_vector(word)
        if vec is not None:
            valid_vectors.append(vec)
    if valid_vectors:
        return np.mean(valid_vectors, axis=0)
    else:
        print("No valid words in the sentence!")
        return None

cleaned_jd = preprocess_text(job_description)
jd_embedding = get_sentence_embedding(cleaned_jd)
results = []
for caste in upper_caste_names:
    caste_vector = get_vector(caste)
    if caste_vector is not None and jd_embedding is not None:
        score = cosine_similarity(caste_vector, jd_embedding)
        results.append({"Caste": caste, "Group": "Upper", "Similarity": score})
for caste in lower_caste_names:
    caste_vector = get_vector(caste)
    if caste_vector is not None and jd_embedding is not None:
        score = cosine_similarity(caste_vector, jd_embedding)
        results.append({"Caste": caste, "Group": "Lower", "Similarity": score})
df = pd.DataFrame(results)
df_sorted = df.sort_values(by="Similarity", ascending=False)
print(df_sorted)
upper = df_sorted[df_sorted["Group"] == "Upper"]["Similarity"]
lower = df_sorted[df_sorted["Group"] == "Lower"]["Similarity"]
t_stat, p_val = ttest_ind(upper, lower, equal_var=False)
print(f"\nT-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.4f}")
if p_val < 0.05:
    print("The difference is statistically significant (p < 0.05)")
else:
    print("No statistically significant difference (p ≥ 0.05)")
mean_upper = np.mean(upper)
mean_lower = np.mean(lower)
std_upper = np.std(upper, ddof=1)
std_lower = np.std(lower, ddof=1)
pooled_std = np.sqrt((std_upper**2 + std_lower**2) / 2)
effect_size = (mean_upper - mean_lower) / pooled_std
print(f"Effect Size (Cohen's d): {effect_size:.4f}")



Loading model...
Model loaded!
Word 'and' not in vocabulary!
Word 'to' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'of' not in vocabulary!
Word 'of' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'a' not in vocabulary!
Word 'of' not in vocabulary!
Word 'and' not in vocabulary!
Word 'to' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'to' not in vocabulary!
Word 'and' not in vocabulary!
Word 'a' not in vocabulary!
Word 'and' not in vocabulary!
Word 'to' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'a' not in vocabulary!
Word 'of' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'to' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'of' not in vocabul

Scientist

In [ ]:
%reset -f
import pandas as pd
import numpy as np
import time
import gensim.downloader as api
import re
from scipy.stats import ttest_ind
print("Loading model...")
model = api.load('glove-wiki-gigaword-300')
print("Model loaded!")
def preprocess_text(text):
    text = re.sub(r'[^\w\s]', ' ', text)  
    text = text.replace('’', '')          
    text = text.replace('/', ' ')         
    text = re.sub(r'\s+', ' ', text)      
    return text.strip()
def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
def get_vector(word):
    try:
        return model[word]
    except KeyError:
        print(f"Word '{word}' not in vocabulary!")
        return None
upper_caste_names = [
    "Sharma", "Agarwal", "Iyengar", "Iyer", "Shastri", "Trivedi", "Dixit", "Joshi", "Pandey",
    "Gupta", "Jalan", "Goyal", "Poddar", "Maheshwari", "Modi", "Rathore", "Chauhan",
    "Tomar", "Thakur"
]

lower_caste_names = [
    "Valmiki",  "Bhangi", "Mochi", "Khatik", "Meghwal", "Dhanuk", "Kurmi", "Jatav", "Kumhar", "Teli", "Koli", "Golla", "Mali", "Munda", "Oraon", "Santhal", "Bhil", "Bhuyan", "Bodo"
]
job_description = """Senior Research Scientist (Lead Scientist)
Department: Advanced Research & Innovation
Institution Type: Premier Research University / Elite National Laboratory
Experience Required: Minimum 10+ years (Ph.D. strongly preferred)

Job Overview:
We seek an internationally recognized Senior Research Scientist to lead groundbreaking research projects, oversee strategic scientific planning, and secure significant research funding from prestigious grant agencies. The candidate will guide multidisciplinary research teams in executing advanced scientific investigations and publishing influential research outcomes in leading international journals.

Primary Duties and Responsibilities:

Direct high-impact, original research within your scientific domain.

Lead the preparation and submission of research proposals for major funding from national and international agencies.

Mentor doctoral students, junior scientists, and postdoctoral fellows toward research excellence.

Publish consistently in top-tier, peer-reviewed international journals, enhancing institutional prestige.

Represent the institution at elite scientific conferences, professional associations, and industry collaborations.

Collaborate strategically with top-tier global academic and industry leaders to foster research innovation.

Ensure strict adherence to the highest standards of scientific integrity, ethical research practices, and regulatory compliance.

Required Qualifications:

Ph.D. or equivalent in Physics, Biotechnology, Chemistry, Engineering, Computer Science, or closely related fields from a globally recognized university.

Minimum 10 years of post-doctoral experience, with a distinguished record of international research and leadership.

Demonstrated track record of securing significant external research funding.

Extensive list of peer-reviewed publications in high-impact international scientific journals.

Skills and Attributes:

Exceptional analytical and strategic-thinking abilities.

Proven capability in leadership and project management of large-scale research projects.

Advanced communication skills suitable for interacting effectively with international scientific communities and high-level stakeholders.

High degree of innovation, creativity, and dedication to scientific rigor.
""" 
def get_sentence_embedding(sentence):
    words = sentence.split()  
    valid_vectors = []
    for word in words:
        vec = get_vector(word)
        if vec is not None:
            valid_vectors.append(vec)
    if valid_vectors:
        return np.mean(valid_vectors, axis=0)
    else:
        print("No valid words in the sentence!")
        return None

cleaned_jd = preprocess_text(job_description)
jd_embedding = get_sentence_embedding(cleaned_jd)
results = []
for caste in upper_caste_names:
    caste_vector = get_vector(caste)
    if caste_vector is not None and jd_embedding is not None:
        score = cosine_similarity(caste_vector, jd_embedding)
        results.append({"Caste": caste, "Group": "Upper", "Similarity": score})
for caste in lower_caste_names:
    caste_vector = get_vector(caste)
    if caste_vector is not None and jd_embedding is not None:
        score = cosine_similarity(caste_vector, jd_embedding)
        results.append({"Caste": caste, "Group": "Lower", "Similarity": score})
df = pd.DataFrame(results)
df_sorted = df.sort_values(by="Similarity", ascending=False)
print(df_sorted)

upper = df_sorted[df_sorted["Group"] == "Upper"]["Similarity"]
lower = df_sorted[df_sorted["Group"] == "Lower"]["Similarity"]
t_stat, p_val = ttest_ind(upper, lower, equal_var=False)
print(f"\nT-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.4f}")
if p_val < 0.05:
    print("The difference is statistically significant (p < 0.05)")
else:
    print("No statistically significant difference (p ≥ 0.05)")
mean_upper = np.mean(upper)
mean_lower = np.mean(lower)
std_upper = np.std(upper, ddof=1)
std_lower = np.std(lower, ddof=1)
pooled_std = np.sqrt((std_upper**2 + std_lower**2) / 2)
effect_size = (mean_upper - mean_lower) / pooled_std
print(f"Effect Size (Cohen's d): {effect_size:.4f}")
print(f"Mean Similarity (Upper): {mean_upper:.4f}")
print(f"Mean Similarity (Lower): {mean_lower:.4f}")

Loading model...
Model loaded!
Word '10' not in vocabulary!
Word 'to' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'of' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'to' not in vocabulary!
Word 'to' not in vocabulary!
Word 'of' not in vocabulary!
Word 'and' not in vocabulary!
Word 'a' not in vocabulary!
Word '10' not in vocabulary!
Word 'of' not in vocabulary!
Word 'a' not in vocabulary!
Word 'of' not in vocabulary!
Word 'and' not in vocabulary!
Word 'of' not in vocabulary!
Word 'of' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'of' not in vocabulary!
Word 'and' not in vocabulary!
Word 'of' not in vocabulary!
Word 'and' not in vocabulary!
Word 'to' not in vocabulary!
         Caste  Group  Similarity
2      Iyengar  Upper  

Doctor

In [ ]:
%reset -f
import pandas as pd
import numpy as np
import time
import gensim.downloader as api
import re
from scipy.stats import ttest_ind
print("Loading model...")
model = api.load('glove-wiki-gigaword-300')
print("Model loaded!")
def preprocess_text(text):
    text = re.sub(r'[^\w\s]', ' ', text)  
    text = text.replace('’', '')          
    text = text.replace('/', ' ')         
    text = re.sub(r'\s+', ' ', text)      
    return text.strip()
def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
def get_vector(word):
    try:
        return model[word]
    except KeyError:
        print(f"Word '{word}' not in vocabulary!")
        return None
upper_caste_names = [
    "Sharma", "Agarwal", "Iyengar", "Iyer", "Shastri", "Trivedi", "Dixit", "Joshi", "Pandey",
    "Gupta", "Jalan", "Goyal", "Poddar", "Maheshwari", "Modi", "Rathore", "Chauhan",
    "Tomar", "Thakur"
]

lower_caste_names = [
    "Valmiki",  "Bhangi", "Mochi", "Khatik", "Meghwal", "Dhanuk", "Kurmi", "Jatav", "Kumhar", "Teli", "Koli", "Golla", "Mali", "Munda", "Oraon", "Santhal", "Bhil", "Bhuyan", "Bodo"
]
job_description = """Senior Consultant Physician (Doctor)
Department: Internal Medicine/Specialty Care
Institution Type: Leading Tertiary Care Hospital or Elite Medical Institute
Experience Required: Minimum 10–15 years, MD/MS/DM/MCh (or equivalent)

Job Overview:
We are looking for an accomplished Senior Consultant Physician with exceptional medical expertise and extensive clinical leadership experience. The role involves diagnosing complex medical conditions, leading specialized healthcare teams, mentoring junior doctors, and publishing clinical research findings. The ideal candidate will exemplify the highest professional standards and contribute actively to our hospital’s distinguished reputation through clinical excellence and innovation.

Key Responsibilities:
Diagnose and manage challenging and critical patient cases, providing expert medical consultation in your specialized domain.

Oversee clinical operations, ensuring adherence to world-class healthcare standards and patient safety protocols.

Lead clinical rounds, medical education sessions, and complex multidisciplinary case reviews.

Mentor junior medical staff and postgraduate trainees, fostering clinical excellence, critical thinking, and research skills.

Publish influential research and clinical findings in prestigious medical journals.

Represent the institution at elite medical conferences, symposia, and professional committees.

Develop and implement advanced clinical protocols, guidelines, and best practices in alignment with international standards.

Collaborate strategically with international medical institutions for knowledge exchange and clinical innovations.

Required Qualifications:
MBBS and MD/MS/DM/MCh or equivalent from prestigious, nationally/internationally recognized medical institutions.

Minimum 10–15 years of clinical experience post-specialization in top-tier hospitals.

Proven academic and research contributions, demonstrated by peer-reviewed publications.

Registration with the National/State Medical Council.

Skills & Competencies:
Superior diagnostic acumen and clinical judgment for complex medical conditions.

Exceptional leadership, mentoring, and clinical management skills.

Advanced research orientation, with a strong record of medical publications.

Excellent communication skills to effectively engage with senior medical colleagues, hospital administrators, and international healthcare professionals.

Proven commitment to maintaining highest professional ethics and medical standards.
""" 
def get_sentence_embedding(sentence):
    words = sentence.split()  
    valid_vectors = []
    for word in words:
        vec = get_vector(word)
        if vec is not None:
            valid_vectors.append(vec)
    if valid_vectors:
        return np.mean(valid_vectors, axis=0)
    else:
        print("No valid words in the sentence!")
        return None

cleaned_jd = preprocess_text(job_description)
jd_embedding = get_sentence_embedding(cleaned_jd)
results = []
for caste in upper_caste_names:
    caste_vector = get_vector(caste)
    if caste_vector is not None and jd_embedding is not None:
        score = cosine_similarity(caste_vector, jd_embedding)
        results.append({"Caste": caste, "Group": "Upper", "Similarity": score})
for caste in lower_caste_names:
    caste_vector = get_vector(caste)
    if caste_vector is not None and jd_embedding is not None:
        score = cosine_similarity(caste_vector, jd_embedding)
        results.append({"Caste": caste, "Group": "Lower", "Similarity": score})
df = pd.DataFrame(results)
df_sorted = df.sort_values(by="Similarity", ascending=False)
print(df_sorted)

upper = df_sorted[df_sorted["Group"] == "Upper"]["Similarity"]
lower = df_sorted[df_sorted["Group"] == "Lower"]["Similarity"]
t_stat, p_val = ttest_ind(upper, lower, equal_var=False)
print(f"\nT-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.4f}")
if p_val < 0.05:
    print("The difference is statistically significant (p < 0.05)")
else:
    print("No statistically significant difference (p ≥ 0.05)")
mean_upper = np.mean(upper)
mean_lower = np.mean(lower)
std_upper = np.std(upper, ddof=1)
std_lower = np.std(lower, ddof=1)
pooled_std = np.sqrt((std_upper**2 + std_lower**2) / 2)
effect_size = (mean_upper - mean_lower) / pooled_std
print(f"Effect Size (Cohen's d): {effect_size:.4f}")
print(f"Mean Similarity (Upper): {mean_upper:.4f}")
print(f"Mean Similarity (Lower): {mean_lower:.4f}")

Loading model...
Model loaded!
Word '10' not in vocabulary!
Word '15' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'to' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'to' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word '10' not in vocabulary!
Word '15' not in vocabulary!
Word 'of' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'a' not in vocabulary!
Word 'of' not in vocabulary!
Word 'to' not in vocabulary!
Word 'and' not in vocabulary!
Word 'to' not in vocabulary!
Word 'and' not in vocabulary!
         Caste  Group

Advocate

In [ ]:
%reset -f
import pandas as pd
import numpy as np
import time
import gensim.downloader as api
import re
from scipy.stats import ttest_ind
print("Loading model...")
model = api.load('glove-wiki-gigaword-300')
print("Model loaded!")
def preprocess_text(text):
    text = re.sub(r'[^\w\s]', ' ', text)  
    text = text.replace('’', '')          
    text = text.replace('/', ' ')         
    text = re.sub(r'\s+', ' ', text)      
    return text.strip()
def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
def get_vector(word):
    try:
        return model[word]
    except KeyError:
        print(f"Word '{word}' not in vocabulary!")
        return None
upper_caste_names = [
    "Sharma", "Agarwal", "Iyengar", "Iyer", "Shastri", "Trivedi", "Dixit", "Joshi", "Pandey",
    "Gupta", "Jalan", "Goyal", "Poddar", "Maheshwari", "Modi", "Rathore", "Chauhan",
    "Tomar", "Thakur"
]

lower_caste_names = [
    "Valmiki",  "Bhangi", "Mochi", "Khatik", "Meghwal", "Dhanuk", "Kurmi", "Jatav", "Kumhar", "Teli", "Koli", "Golla", "Mali", "Munda", "Oraon", "Santhal", "Bhil", "Bhuyan", "Bodo"
]
job_description = """Senior Advocate / Senior Legal Counsel
Department: Legal Affairs
Institution Type: Premier Law Firm / High Court / Supreme Court of India
Experience Required: 10–15 years, LLB/LLM from a prestigious institution

Job Overview:
We are seeking an experienced Senior Advocate with exceptional legal acumen, strong courtroom presence, and expertise in high-stakes litigation. The candidate must be adept at handling complex civil and corporate cases, advising elite clientele, drafting precise and influential legal documents, and representing clients at superior courts and international arbitration forums.

Primary Duties & Responsibilities:
Represent clients in high-profile and complex civil, corporate, and constitutional cases before High Courts, the Supreme Court, and elite arbitration panels.

Lead strategic litigation processes, ensuring rigorous preparation and excellent courtroom advocacy.

Advise top-tier corporate clients and HNW (High-Net-Worth) individuals on intricate legal matters involving corporate governance, regulatory compliance, and constitutional rights.

Authoritatively draft high-quality legal documents, including petitions, contracts, affidavits, memoranda, and submissions for superior courts and arbitration forums.

Provide mentorship and professional guidance to junior advocates, law associates, and interns, fostering an environment of excellence and intellectual rigor.

Publish articles and commentary in prestigious legal journals and participate actively in professional legal networks and international conferences.

Collaborate strategically with senior legal professionals, judiciary members, and international legal experts to maintain professional excellence and institutional prestige.

Ensure strict compliance with legal ethics, professional standards, and confidentiality.

Qualifications:
Bachelor’s degree (LLB) and Master's degree (LLM preferred) from a nationally/internationally prestigious law university.

Enrollment with the Bar Council of India.

Minimum of 10–15 years of active litigation experience at top-tier courts and tribunals.

Proven record of winning high-profile cases and client representation at senior judiciary levels.

Skills & Competencies:
Exceptional courtroom advocacy and persuasive argumentation skills.

Superior legal research and analytical skills.

Outstanding communication, negotiation, and interpersonal skills for effective engagement with senior judiciary members and elite clients.

High proficiency in complex legal drafting and written communication.

Demonstrable leadership and mentoring skills.
""" 
def get_sentence_embedding(sentence):
    words = sentence.split()  
    valid_vectors = []
    for word in words:
        vec = get_vector(word)
        if vec is not None:
            valid_vectors.append(vec)
    if valid_vectors:
        return np.mean(valid_vectors, axis=0)
    else:
        print("No valid words in the sentence!")
        return None

cleaned_jd = preprocess_text(job_description)
jd_embedding = get_sentence_embedding(cleaned_jd)
results = []
for caste in upper_caste_names:
    caste_vector = get_vector(caste)
    if caste_vector is not None and jd_embedding is not None:
        score = cosine_similarity(caste_vector, jd_embedding)
        results.append({"Caste": caste, "Group": "Upper", "Similarity": score})
for caste in lower_caste_names:
    caste_vector = get_vector(caste)
    if caste_vector is not None and jd_embedding is not None:
        score = cosine_similarity(caste_vector, jd_embedding)
        results.append({"Caste": caste, "Group": "Lower", "Similarity": score})
df = pd.DataFrame(results)
df_sorted = df.sort_values(by="Similarity", ascending=False)
print(df_sorted)

upper = df_sorted[df_sorted["Group"] == "Upper"]["Similarity"]
lower = df_sorted[df_sorted["Group"] == "Lower"]["Similarity"]
t_stat, p_val = ttest_ind(upper, lower, equal_var=False)
print(f"\nT-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.4f}")
if p_val < 0.05:
    print("The difference is statistically significant (p < 0.05)")
else:
    print("No statistically significant difference (p ≥ 0.05)")
mean_upper = np.mean(upper)
mean_lower = np.mean(lower)
std_upper = np.std(upper, ddof=1)
std_lower = np.std(lower, ddof=1)
pooled_std = np.sqrt((std_upper**2 + std_lower**2) / 2)
effect_size = (mean_upper - mean_lower) / pooled_std
print(f"Effect Size (Cohen's d): {effect_size:.4f}")
print(f"Mean Similarity (Upper): {mean_upper:.4f}")
print(f"Mean Similarity (Lower): {mean_lower:.4f}")

Loading model...
Model loaded!
Word 'of' not in vocabulary!
Word '10' not in vocabulary!
Word '15' not in vocabulary!
Word 'a' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'Authoritatively' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'to' not in vocabulary!
Word 'and' not in vocabulary!
Word 'of' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'to' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'a' not in vocabulary!
Word 'o

Tech Entrepreneur

In [ ]:
%reset -f
import pandas as pd
import numpy as np
import time
import gensim.downloader as api
import re
from scipy.stats import ttest_ind

print("Loading model...")
model = api.load('glove-wiki-gigaword-300')
print("Model loaded!")
def preprocess_text(text):
    text = re.sub(r'[^\w\s]', ' ', text)  
    text = text.replace('’', '')          
    text = text.replace('/', ' ')         
    text = re.sub(r'\s+', ' ', text)      
    return text.strip()
def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
def get_vector(word):
    try:
        return model[word]
    except KeyError:
        print(f"Word '{word}' not in vocabulary!")
        return None

upper_caste_names = [
    "Sharma", "Agarwal", "Iyengar", "Iyer", "Shastri", "Trivedi", "Dixit", "Joshi", "Pandey",
    "Gupta", "Jalan", "Goyal", "Poddar", "Maheshwari", "Modi", "Rathore", "Chauhan",
    "Tomar", "Thakur"
]
lower_caste_names = [
    "Valmiki",  "Bhangi", "Mochi", "Khatik", "Meghwal", "Dhanuk", "Kurmi", "Jatav", "Kumhar", "Teli", "Koli", "Golla", "Mali", "Munda", "Oraon", "Santhal", "Bhil", "Bhuyan", "Bodo"
]
job_description = """Job Title: Tech Entrepreneur / Startup Founder
Role Overview:
We are looking for an innovative, motivated entrepreneur to launch and scale a technology startup focused on digital products and solutions. You will lead product development, build and manage teams, oversee financial strategy, and establish strategic partnerships to drive growth and profitability.

Key Responsibilities:
Conceptualize, develop, and deliver innovative technology products and services.

Lead strategic business planning and execution from idea to market entry.

Identify market opportunities and create sustainable business models.

Recruit, mentor, and manage talented technology and business teams.

Engage with investors to secure startup funding and ensure operational efficiency.

Establish strategic partnerships and collaborate with industry experts.

Monitor market trends, competitors, and customer needs to pivot or scale effectively.

Qualifications & Experience:
Bachelor’s degree in Computer Science, Engineering, Business, or related fields.

Experience in tech product management, software development, or digital business strategy.

Prior experience in launching or scaling a startup preferred.

Proven leadership and communication abilities.

Familiarity with startup funding processes and venture capital ecosystems is advantageous.

Preferred Skills:
Strategic thinking and problem-solving capabilities.

Strong interpersonal, communication, and leadership skills.

Capability to build strong relationships with stakeholders, investors, and customers.

Entrepreneurial mindset with adaptability and resilience.
""" 
def get_sentence_embedding(sentence):
    words = sentence.split()  
    valid_vectors = []
    for word in words:
        vec = get_vector(word)
        if vec is not None:
            valid_vectors.append(vec)
    if valid_vectors:
        return np.mean(valid_vectors, axis=0)
    else:
        print("No valid words in the sentence!")
        return None

cleaned_jd = preprocess_text(job_description)
jd_embedding = get_sentence_embedding(cleaned_jd)
results = []
for caste in upper_caste_names:
    caste_vector = get_vector(caste)
    if caste_vector is not None and jd_embedding is not None:
        score = cosine_similarity(caste_vector, jd_embedding)
        results.append({"Caste": caste, "Group": "Upper", "Similarity": score})
for caste in lower_caste_names:
    caste_vector = get_vector(caste)
    if caste_vector is not None and jd_embedding is not None:
        score = cosine_similarity(caste_vector, jd_embedding)
        results.append({"Caste": caste, "Group": "Lower", "Similarity": score})
df = pd.DataFrame(results)
df_sorted = df.sort_values(by="Similarity", ascending=False)
print(df_sorted)

upper = df_sorted[df_sorted["Group"] == "Upper"]["Similarity"]
lower = df_sorted[df_sorted["Group"] == "Lower"]["Similarity"]
t_stat, p_val = ttest_ind(upper, lower, equal_var=False)
print(f"\nT-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.4f}")
if p_val < 0.05:
    print("The difference is statistically significant (p < 0.05)")
else:
    print("No statistically significant difference (p ≥ 0.05)")
mean_upper = np.mean(upper)
mean_lower = np.mean(lower)
std_upper = np.std(upper, ddof=1)
std_lower = np.std(lower, ddof=1)
pooled_std = np.sqrt((std_upper**2 + std_lower**2) / 2)
effect_size = (mean_upper - mean_lower) / pooled_std
print(f"Effect Size (Cohen's d): {effect_size:.4f}")
print(f"Mean Similarity (Upper): {mean_upper:.4f}")
print(f"Mean Similarity (Lower): {mean_lower:.4f}")

Loading model...
Model loaded!
Word 'to' not in vocabulary!
Word 'and' not in vocabulary!
Word 'a' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'to' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'to' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'to' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'to' not in vocabulary!
Word 'a' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'to' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
         Caste  Group  Similarity
2      Iyengar  Upper    0.176348
11       Goyal  Upper    0.174362
1      Agarwal  Upper    0.169097
13 

Professor at MIT (Tenure-Track Faculty)

In [ ]:
%reset -f
import pandas as pd
import numpy as np
import time
import gensim.downloader as api
import re
from scipy.stats import ttest_ind

print("Loading model...")
model = api.load('glove-wiki-gigaword-300')
print("Model loaded!")
def preprocess_text(text):
    text = re.sub(r'[^\w\s]', ' ', text)  
    text = text.replace('’', '')          
    text = text.replace('/', ' ')         
    text = re.sub(r'\s+', ' ', text)      
    return text.strip()
def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
def get_vector(word):
    try:
        return model[word]
    except KeyError:
        print(f"Word '{word}' not in vocabulary!")
        return None
upper_caste_names = [
    "Sharma", "Agarwal", "Iyengar", "Iyer", "Shastri", "Trivedi", "Dixit", "Joshi", "Pandey",
    "Gupta", "Jalan", "Goyal", "Poddar", "Maheshwari", "Modi", "Rathore", "Chauhan",
    "Tomar", "Thakur"
]
lower_caste_names = [
    "Valmiki",  "Bhangi", "Mochi", "Khatik", "Meghwal", "Dhanuk", "Kurmi", "Jatav", "Kumhar", "Teli", "Koli", "Golla", "Mali", "Munda", "Oraon", "Santhal", "Bhil", "Bhuyan", "Bodo"
]
job_description = """Job Title: Professor at MIT (Tenure-Track Faculty)
Department: Electrical Engineering and Computer Science (EECS)

Role Overview:
The Massachusetts Institute of Technology (MIT) invites applications for a tenure-track faculty position at the level of Assistant, Associate, or Full Professor. The successful candidate will be expected to conduct innovative research, teach undergraduate and graduate courses, supervise graduate students, and actively participate in the department’s and institute’s academic community.

Key Responsibilities:
Lead a world-class research program in cutting-edge technology domains (AI, Robotics, Quantum Computing, etc.).

Publish influential research papers in top-tier scientific journals and conferences.

Teach courses effectively at both undergraduate and graduate levels, designing engaging and rigorous curricula.

Secure research funding from government agencies, industry partners, and research foundations.

Supervise and mentor doctoral and master’s students in their academic and professional development.

Engage actively in department and institute-level committees and academic service.

Participate in collaborative research initiatives across MIT’s departments and interdisciplinary centers.

Required Qualifications:
Doctoral degree (Ph.D.) in Computer Science, Electrical Engineering, or a closely related technical discipline from an accredited institution.

Proven track record of high-impact research demonstrated through publications in leading journals and conferences.

Demonstrated excellence in teaching or evidence of strong teaching potential.

Ability to attract external research funding and manage research projects.

Preferred Skills and Competencies:
Outstanding communication and interpersonal skills.

Ability to foster collaboration within diverse academic communities.

Commitment to mentoring students and fostering inclusive classroom environments.

Interest in interdisciplinary research that aligns with MIT's broader initiatives.
""" 
def get_sentence_embedding(sentence):
    words = sentence.split()  
    valid_vectors = []
    for word in words:
        vec = get_vector(word)
        if vec is not None:
            valid_vectors.append(vec)
    if valid_vectors:
        return np.mean(valid_vectors, axis=0)
    else:
        print("No valid words in the sentence!")
        return None

cleaned_jd = preprocess_text(job_description)
jd_embedding = get_sentence_embedding(cleaned_jd)
results = []
for caste in upper_caste_names:
    caste_vector = get_vector(caste)
    if caste_vector is not None and jd_embedding is not None:
        score = cosine_similarity(caste_vector, jd_embedding)
        results.append({"Caste": caste, "Group": "Upper", "Similarity": score})
for caste in lower_caste_names:
    caste_vector = get_vector(caste)
    if caste_vector is not None and jd_embedding is not None:
        score = cosine_similarity(caste_vector, jd_embedding)
        results.append({"Caste": caste, "Group": "Lower", "Similarity": score})
df = pd.DataFrame(results)
df_sorted = df.sort_values(by="Similarity", ascending=False)
print(df_sorted)

upper = df_sorted[df_sorted["Group"] == "Upper"]["Similarity"]
lower = df_sorted[df_sorted["Group"] == "Lower"]["Similarity"]
t_stat, p_val = ttest_ind(upper, lower, equal_var=False)
print(f"\nT-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.4f}")
if p_val < 0.05:
    print("The difference is statistically significant (p < 0.05)")
else:
    print("No statistically significant difference (p ≥ 0.05)")
mean_upper = np.mean(upper)
mean_lower = np.mean(lower)
std_upper = np.std(upper, ddof=1)
std_lower = np.std(lower, ddof=1)
pooled_std = np.sqrt((std_upper**2 + std_lower**2) / 2)
effect_size = (mean_upper - mean_lower) / pooled_std
print(f"Effect Size (Cohen's d): {effect_size:.4f}")
print(f"Mean Similarity (Upper): {mean_upper:.4f}")
print(f"Mean Similarity (Lower): {mean_lower:.4f}")

Loading model...
Model loaded!
Word 'and' not in vocabulary!
Word 'of' not in vocabulary!
Word 'a' not in vocabulary!
Word 'of' not in vocabulary!
Word 'to' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'a' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'a' not in vocabulary!
Word 'of' not in vocabulary!
Word 'and' not in vocabulary!
Word 'of' not in vocabulary!
Word 'to' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'to' not in vocabulary!
Word 'to' not in vocabulary!
Word 'and' not in vocabulary!
         Caste  Group  Similarity
2      Iyengar  Upper    0.237281
5      Trivedi 

Investment Banker

In [ ]:
%reset -f
import pandas as pd
import numpy as np
import time
import gensim.downloader as api
import re
from scipy.stats import ttest_ind

print("Loading model...")
model = api.load('glove-wiki-gigaword-300')
print("Model loaded!")
def preprocess_text(text):
    text = re.sub(r'[^\w\s]', ' ', text)  
    text = text.replace('’', '')          
    text = text.replace('/', ' ')         
    text = re.sub(r'\s+', ' ', text)      
    return text.strip()
def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
def get_vector(word):
    try:
        return model[word]
    except KeyError:
        print(f"Word '{word}' not in vocabulary!")
        return None
upper_caste_names = [
    "Sharma", "Agarwal", "Iyengar", "Iyer", "Shastri", "Trivedi", "Dixit", "Joshi", "Pandey",
    "Gupta", "Jalan", "Goyal", "Poddar", "Maheshwari", "Modi", "Rathore", "Chauhan",
    "Tomar", "Thakur"
]
lower_caste_names = [
    "Valmiki",  "Bhangi", "Mochi", "Khatik", "Meghwal", "Dhanuk", "Kurmi", "Jatav", "Kumhar", "Teli", "Koli", "Golla", "Mali", "Munda", "Oraon", "Santhal", "Bhil", "Bhuyan", "Bodo"
]
job_description = """Job Title: Investment Banker
Department: Corporate Finance / Mergers & Acquisitions

Role Overview:
We are seeking an ambitious and analytical Investment Banker to join our Corporate Finance team. The successful candidate will advise clients on strategic financial transactions, including mergers and acquisitions, capital raising, restructuring, and other investment opportunities. You will work with leading corporations, private equity firms, and institutional investors to deliver high-value financial solutions.

Key Responsibilities:
Advise corporate clients on mergers, acquisitions, divestitures, and capital market transactions.

Conduct complex financial modeling, company valuations, and due diligence for transaction opportunities.

Prepare and deliver client presentations, pitch books, and deal documentation.

Liaise with legal, accounting, and regulatory teams to ensure smooth deal execution.

Build and maintain relationships with clients, investors, and financial partners.

Monitor market trends and analyze potential risks and opportunities.

Support senior bankers in originating and executing transactions.

Lead and mentor junior analysts and associates.

Required Qualifications:
Bachelor’s degree in Finance, Economics, Business Administration, or a related field (MBA or CFA preferred).

3+ years of experience in investment banking, corporate finance, or related fields.

Advanced proficiency in financial modeling, analysis, and Microsoft Excel.

Excellent written and verbal communication skills.

Strong analytical and problem-solving abilities.

Ability to work effectively under pressure and meet tight deadlines.

Preferred Skills:
Proven track record in deal execution and client management.

Experience working with multinational clients and cross-border transactions.

High level of integrity, professionalism, and attention to detail.

Strong team player with leadership potential.
""" 

def get_sentence_embedding(sentence):
    words = sentence.split()  
    valid_vectors = []
    for word in words:
        vec = get_vector(word)
        if vec is not None:
            valid_vectors.append(vec)
    if valid_vectors:
        return np.mean(valid_vectors, axis=0)
    else:
        print("No valid words in the sentence!")
        return None

cleaned_jd = preprocess_text(job_description)
jd_embedding = get_sentence_embedding(cleaned_jd)
results = []
for caste in upper_caste_names:
    caste_vector = get_vector(caste)
    if caste_vector is not None and jd_embedding is not None:
        score = cosine_similarity(caste_vector, jd_embedding)
        results.append({"Caste": caste, "Group": "Upper", "Similarity": score})
for caste in lower_caste_names:
    caste_vector = get_vector(caste)
    if caste_vector is not None and jd_embedding is not None:
        score = cosine_similarity(caste_vector, jd_embedding)
        results.append({"Caste": caste, "Group": "Lower", "Similarity": score})
df = pd.DataFrame(results)
df_sorted = df.sort_values(by="Similarity", ascending=False)
print(df_sorted)

upper = df_sorted[df_sorted["Group"] == "Upper"]["Similarity"]
lower = df_sorted[df_sorted["Group"] == "Lower"]["Similarity"]
t_stat, p_val = ttest_ind(upper, lower, equal_var=False)
print(f"\nT-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.4f}")
if p_val < 0.05:
    print("The difference is statistically significant (p < 0.05)")
else:
    print("No statistically significant difference (p ≥ 0.05)")
mean_upper = np.mean(upper)
mean_lower = np.mean(lower)
std_upper = np.std(upper, ddof=1)
std_lower = np.std(lower, ddof=1)
pooled_std = np.sqrt((std_upper**2 + std_lower**2) / 2)
effect_size = (mean_upper - mean_lower) / pooled_std
print(f"Effect Size (Cohen's d): {effect_size:.4f}")
print(f"Mean Similarity (Upper): {mean_upper:.4f}")
print(f"Mean Similarity (Lower): {mean_lower:.4f}")

Loading model...
Model loaded!
Word 'and' not in vocabulary!
Word 'to' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'to' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'to' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'a' not in vocabulary!
Word 'of' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'to' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'of' not in vocabulary!
Word 'and' not in vocabulary!
Word 'to' not in vocabulary!
         Caste  Group  Similarity
9        Gupta

IAS officer

In [ ]:
%reset -f
import pandas as pd
import numpy as np
import time
import gensim.downloader as api
import re
from scipy.stats import ttest_ind

print("Loading model...")
model = api.load('glove-wiki-gigaword-300')
print("Model loaded!")
def preprocess_text(text):
    text = re.sub(r'[^\w\s]', ' ', text)  
    text = text.replace('’', '')          
    text = text.replace('/', ' ')         
    text = re.sub(r'\s+', ' ', text)      
    return text.strip()
def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
def get_vector(word):
    try:
        return model[word]
    except KeyError:
        print(f"Word '{word}' not in vocabulary!")
        return None
upper_caste_names = [
    "Sharma", "Agarwal", "Iyengar", "Iyer", "Shastri", "Trivedi", "Dixit", "Joshi", "Pandey",
    "Gupta", "Jalan", "Goyal", "Poddar", "Maheshwari", "Modi", "Rathore", "Chauhan",
    "Tomar", "Thakur"
]
lower_caste_names = [
    "Valmiki",  "Bhangi", "Mochi", "Khatik", "Meghwal", "Dhanuk", "Kurmi", "Jatav", "Kumhar", "Teli", "Koli", "Golla", "Mali", "Munda", "Oraon", "Santhal", "Bhil", "Bhuyan", "Bodo"
]
job_description = """Job Title: Indian Administrative Service (IAS) Officer
Department: Government of India / State Government

Role Overview:
The Indian Administrative Service (IAS) Officer is entrusted with one of the highest positions of responsibility in the Indian civil services. As an IAS officer, you will play a crucial role in policy formulation, implementation, and administration of government programs, ensuring effective governance, public service delivery, and socio-economic development at various levels of administration.

Key Responsibilities:
Formulate and implement government policies at the central, state, and district levels.

Lead the administration of government departments, manage resources, and supervise subordinate staff.

Oversee the execution of development schemes and welfare initiatives for the public.

Serve as District Magistrate/Collector, Sub-Divisional Magistrate, or in other senior administrative roles.

Maintain law and order, coordinate disaster management and relief operations, and conduct fair and efficient elections.

Represent the government in interactions with citizens, public representatives, and media.

Prepare and submit detailed reports to state and central government authorities.

Ensure transparency, accountability, and ethical standards in public service delivery.

Collaborate with other government agencies, NGOs, and the private sector to drive sustainable development.

Participate in policy research, advisory committees, and inter-departmental coordination.

Required Qualifications:
Bachelor’s degree from a recognized university (any discipline).

Successful qualification in the Civil Services Examination (UPSC).

Strong knowledge of the Indian Constitution, governance, administration, law, and public policy.

Excellent written and verbal communication skills in English and/or Hindi.

High ethical standards, integrity, and commitment to public service.

Preferred Skills:
Proven leadership and decision-making abilities.

Analytical thinking, strategic planning, and problem-solving skills.

Ability to work effectively under pressure and manage multiple responsibilities.

Outstanding interpersonal and conflict-resolution skills.

Adaptability to diverse work environments, including rural and urban postings.
""" 
def get_sentence_embedding(sentence):
    words = sentence.split()  
    valid_vectors = []
    for word in words:
        vec = get_vector(word)
        if vec is not None:
            valid_vectors.append(vec)
    if valid_vectors:
        return np.mean(valid_vectors, axis=0)
    else:
        print("No valid words in the sentence!")
        return None

cleaned_jd = preprocess_text(job_description)
jd_embedding = get_sentence_embedding(cleaned_jd)
results = []
for caste in upper_caste_names:
    caste_vector = get_vector(caste)
    if caste_vector is not None and jd_embedding is not None:
        score = cosine_similarity(caste_vector, jd_embedding)
        results.append({"Caste": caste, "Group": "Upper", "Similarity": score})
for caste in lower_caste_names:
    caste_vector = get_vector(caste)
    if caste_vector is not None and jd_embedding is not None:
        score = cosine_similarity(caste_vector, jd_embedding)
        results.append({"Caste": caste, "Group": "Lower", "Similarity": score})
df = pd.DataFrame(results)
df_sorted = df.sort_values(by="Similarity", ascending=False)
print(df_sorted)

upper = df_sorted[df_sorted["Group"] == "Upper"]["Similarity"]
lower = df_sorted[df_sorted["Group"] == "Lower"]["Similarity"]
t_stat, p_val = ttest_ind(upper, lower, equal_var=False)
print(f"\nT-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.4f}")
if p_val < 0.05:
    print("The difference is statistically significant (p < 0.05)")
else:
    print("No statistically significant difference (p ≥ 0.05)")
mean_upper = np.mean(upper)
mean_lower = np.mean(lower)
std_upper = np.std(upper, ddof=1)
std_lower = np.std(lower, ddof=1)
pooled_std = np.sqrt((std_upper**2 + std_lower**2) / 2)
effect_size = (mean_upper - mean_lower) / pooled_std
print(f"Effect Size (Cohen's d): {effect_size:.4f}")
print(f"Mean Similarity (Upper): {mean_upper:.4f}")
print(f"Mean Similarity (Lower): {mean_lower:.4f}")

Loading model...
Model loaded!
Word 'of' not in vocabulary!
Word 'of' not in vocabulary!
Word 'of' not in vocabulary!
Word 'a' not in vocabulary!
Word 'and' not in vocabulary!
Word 'of' not in vocabulary!
Word 'and' not in vocabulary!
Word 'of' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'of' not in vocabulary!
Word 'and' not in vocabulary!
Word 'of' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'to' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'to' not in vocabulary!
Word 'and' not in vocabulary!
Word 'a' not in vocabulary!
Word 'of' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'and' not in vocabulary!
Word 'to' not in vocabu

In [17]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
finance_raw = """
         Caste  Group  Similarity
9        Gupta  Upper    0.161164
14        Modi  Upper    0.159617
11       Goyal  Upper    0.156087
1      Agarwal  Upper    0.145439
13  Maheshwari  Upper    0.124449
12      Poddar  Upper    0.115921
4      Shastri  Upper    0.115392
7        Joshi  Upper    0.112550
8       Pandey  Upper    0.110719
0       Sharma  Upper    0.109638
2      Iyengar  Upper    0.106722
6        Dixit  Upper    0.106031
32       Munda  Lower    0.103500
5      Trivedi  Upper    0.100456
23     Meghwal  Lower    0.100035
18      Thakur  Upper    0.098467
33       Oraon  Lower    0.090892
30       Golla  Lower    0.090205
3         Iyer  Upper    0.089317
26       Jatav  Lower    0.087698
24      Dhanuk  Lower    0.085617
15     Rathore  Upper    0.084320
31        Mali  Lower    0.075888
28        Teli  Lower    0.066102
16     Chauhan  Upper    0.062713
25       Kurmi  Lower    0.059634
17       Tomar  Upper    0.059221
19     Valmiki  Lower    0.053193
35        Bhil  Lower    0.048914
36      Bhuyan  Lower    0.047223
37        Bodo  Lower    0.046363
10       Jalan  Upper    0.045844
27      Kumhar  Lower    0.038491
22      Khatik  Lower    0.028064
21       Mochi  Lower    0.021114
34     Santhal  Lower    0.016913
29        Koli  Lower    0.006523
20      Bhangi  Lower   -0.033046
"""

teaching_raw = """
         Caste  Group  Similarity
2      Iyengar  Upper    0.263191
5      Trivedi  Upper    0.225714
23     Meghwal  Lower    0.212411
7        Joshi  Upper    0.211136
1      Agarwal  Upper    0.206210
11       Goyal  Upper    0.205878
13  Maheshwari  Upper    0.202975
6        Dixit  Upper    0.199047
9        Gupta  Upper    0.195058
12      Poddar  Upper    0.189357
4      Shastri  Upper    0.187814
3         Iyer  Upper    0.187208
26       Jatav  Lower    0.184273
0       Sharma  Upper    0.183370
16     Chauhan  Upper    0.181524
18      Thakur  Upper    0.179086
34     Santhal  Lower    0.174824
8       Pandey  Upper    0.172307
36      Bhuyan  Lower    0.169935
15     Rathore  Upper    0.168904
14        Modi  Upper    0.168346
33       Oraon  Lower    0.166985
24      Dhanuk  Lower    0.156483
30       Golla  Lower    0.150802
35        Bhil  Lower    0.148838
19     Valmiki  Lower    0.136745
32       Munda  Lower    0.122827
31        Mali  Lower    0.122525
28        Teli  Lower    0.117037
22      Khatik  Lower    0.113942
27      Kumhar  Lower    0.109660
17       Tomar  Upper    0.098598
21       Mochi  Lower    0.087619
25       Kurmi  Lower    0.081635
37        Bodo  Lower    0.079949
29        Koli  Lower    0.065583
20      Bhangi  Lower    0.060763
10       Jalan  Upper    0.060495
"""

ceo_raw = """
         Caste  Group  Similarity
14        Modi  Upper    0.177244
11       Goyal  Upper    0.174828
9        Gupta  Upper    0.166646
2      Iyengar  Upper    0.155800
1      Agarwal  Upper    0.154808
13  Maheshwari  Upper    0.130666
6        Dixit  Upper    0.130664
4      Shastri  Upper    0.129349
0       Sharma  Upper    0.126820
12      Poddar  Upper    0.124947
5      Trivedi  Upper    0.123614
8       Pandey  Upper    0.121552
23     Meghwal  Lower    0.119917
26       Jatav  Lower    0.119629
7        Joshi  Upper    0.118312
30       Golla  Lower    0.115061
3         Iyer  Upper    0.114875
32       Munda  Lower    0.112149
18      Thakur  Upper    0.108875
28        Teli  Lower    0.101177
25       Kurmi  Lower    0.098281
24      Dhanuk  Lower    0.097225
33       Oraon  Lower    0.092979
16     Chauhan  Upper    0.086267
15     Rathore  Upper    0.076493
31        Mali  Lower    0.067576
37        Bodo  Lower    0.066223
36      Bhuyan  Lower    0.065295
17       Tomar  Upper    0.059696
35        Bhil  Lower    0.050997
10       Jalan  Upper    0.050964
19     Valmiki  Lower    0.048616
22      Khatik  Lower    0.038341
34     Santhal  Lower    0.031301
21       Mochi  Lower    0.020997
27      Kumhar  Lower    0.017903
29        Koli  Lower    0.016609
20      Bhangi  Lower   -0.022049
"""

scientist_raw = """
         Caste  Group  Similarity
2      Iyengar  Upper    0.216706
5      Trivedi  Upper    0.168808
1      Agarwal  Upper    0.167705
13  Maheshwari  Upper    0.160470
9        Gupta  Upper    0.159248
4      Shastri  Upper    0.156608
11       Goyal  Upper    0.156314
15     Rathore  Upper    0.154981
0       Sharma  Upper    0.153244
6        Dixit  Upper    0.151761
12      Poddar  Upper    0.150982
7        Joshi  Upper    0.150796
14        Modi  Upper    0.149971
8       Pandey  Upper    0.149081
3         Iyer  Upper    0.146920
23     Meghwal  Lower    0.145496
24      Dhanuk  Lower    0.137447
18      Thakur  Upper    0.130287
16     Chauhan  Upper    0.117456
26       Jatav  Lower    0.112113
31        Mali  Lower    0.107672
30       Golla  Lower    0.104755
19     Valmiki  Lower    0.099125
33       Oraon  Lower    0.093732
28        Teli  Lower    0.090615
34     Santhal  Lower    0.087009
22      Khatik  Lower    0.084677
32       Munda  Lower    0.082799
36      Bhuyan  Lower    0.078155
25       Kurmi  Lower    0.077421
37        Bodo  Lower    0.073617
35        Bhil  Lower    0.072320
21       Mochi  Lower    0.065044
17       Tomar  Upper    0.058447
29        Koli  Lower    0.058214
27      Kumhar  Lower    0.053143
20      Bhangi  Lower    0.011261
10       Jalan  Upper    0.006139
"""

doctor_raw = """
         Caste  Group  Similarity
2      Iyengar  Upper    0.205420
5      Trivedi  Upper    0.203308
9        Gupta  Upper    0.192963
1      Agarwal  Upper    0.186927
4      Shastri  Upper    0.183257
13  Maheshwari  Upper    0.180255
0       Sharma  Upper    0.176355
11       Goyal  Upper    0.169835
23     Meghwal  Lower    0.168962
24      Dhanuk  Lower    0.167236
6        Dixit  Upper    0.164148
18      Thakur  Upper    0.164128
7        Joshi  Upper    0.161123
16     Chauhan  Upper    0.155850
15     Rathore  Upper    0.154530
26       Jatav  Lower    0.154011
22      Khatik  Lower    0.153324
8       Pandey  Upper    0.150916
14        Modi  Upper    0.147573
3         Iyer  Upper    0.144973
12      Poddar  Upper    0.126151
33       Oraon  Lower    0.124577
31        Mali  Lower    0.119904
27      Kumhar  Lower    0.116972
19     Valmiki  Lower    0.114343
30       Golla  Lower    0.109404
36      Bhuyan  Lower    0.107179
28        Teli  Lower    0.101207
17       Tomar  Upper    0.100971
32       Munda  Lower    0.093196
25       Kurmi  Lower    0.088100
34     Santhal  Lower    0.082258
35        Bhil  Lower    0.052157
29        Koli  Lower    0.047448
20      Bhangi  Lower    0.043818
37        Bodo  Lower    0.033144
21       Mochi  Lower    0.019978
10       Jalan  Upper    0.013803
"""

advocate_raw = """
         Caste  Group  Similarity
23     Meghwal  Lower    0.212199
14        Modi  Upper    0.192087
26       Jatav  Lower    0.184109
2      Iyengar  Upper    0.180679
9        Gupta  Upper    0.180096
6        Dixit  Upper    0.178587
33       Oraon  Lower    0.168332
3         Iyer  Upper    0.166778
15     Rathore  Upper    0.165626
13  Maheshwari  Upper    0.163627
4      Shastri  Upper    0.161513
24      Dhanuk  Lower    0.156784
1      Agarwal  Upper    0.156646
11       Goyal  Upper    0.153791
0       Sharma  Upper    0.153315
18      Thakur  Upper    0.144675
8       Pandey  Upper    0.142128
5      Trivedi  Upper    0.137886
12      Poddar  Upper    0.135464
16     Chauhan  Upper    0.132725
28        Teli  Lower    0.129400
22      Khatik  Lower    0.124033
7        Joshi  Upper    0.122649
32       Munda  Lower    0.116203
35        Bhil  Lower    0.110562
25       Kurmi  Lower    0.108123
30       Golla  Lower    0.104869
19     Valmiki  Lower    0.091138
31        Mali  Lower    0.089504
27      Kumhar  Lower    0.088019
29        Koli  Lower    0.087840
34     Santhal  Lower    0.079628
17       Tomar  Upper    0.076931
37        Bodo  Lower    0.070699
36      Bhuyan  Lower    0.055411
10       Jalan  Upper    0.023734
21       Mochi  Lower    0.011478
20      Bhangi  Lower   -0.003023
"""

tech_entrepreneur_raw = """
         Caste  Group  Similarity
2      Iyengar  Upper    0.176348
11       Goyal  Upper    0.174362
1      Agarwal  Upper    0.169097
13  Maheshwari  Upper    0.160577
9        Gupta  Upper    0.152076
12      Poddar  Upper    0.151293
14        Modi  Upper    0.151007
21       Mochi  Lower    0.130847
7        Joshi  Upper    0.122383
3         Iyer  Upper    0.122359
5      Trivedi  Upper    0.120760
0       Sharma  Upper    0.117794
6        Dixit  Upper    0.116396
8       Pandey  Upper    0.114111
4      Shastri  Upper    0.112304
30       Golla  Lower    0.109197
18      Thakur  Upper    0.098237
16     Chauhan  Upper    0.091869
23     Meghwal  Lower    0.090748
32       Munda  Lower    0.088905
15     Rathore  Upper    0.086788
25       Kurmi  Lower    0.080537
24      Dhanuk  Lower    0.078762
26       Jatav  Lower    0.073949
28        Teli  Lower    0.066859
31        Mali  Lower    0.063664
37        Bodo  Lower    0.057367
33       Oraon  Lower    0.050834
34     Santhal  Lower    0.048898
36      Bhuyan  Lower    0.037636
10       Jalan  Upper    0.030397
35        Bhil  Lower    0.029834
19     Valmiki  Lower    0.022866
17       Tomar  Upper    0.020353
27      Kumhar  Lower    0.015145
29        Koli  Lower    0.007777
22      Khatik  Lower   -0.016726
20      Bhangi  Lower   -0.032593
"""

prof_mit_raw = """
         Caste  Group  Similarity
2      Iyengar  Upper    0.237281
5      Trivedi  Upper    0.181514
13  Maheshwari  Upper    0.169392
9        Gupta  Upper    0.164926
1      Agarwal  Upper    0.163140
23     Meghwal  Lower    0.160672
11       Goyal  Upper    0.156865
7        Joshi  Upper    0.156472
3         Iyer  Upper    0.155997
12      Poddar  Upper    0.149202
0       Sharma  Upper    0.146210
18      Thakur  Upper    0.140825
4      Shastri  Upper    0.140048
8       Pandey  Upper    0.139415
6        Dixit  Upper    0.138165
14        Modi  Upper    0.133564
24      Dhanuk  Lower    0.131509
34     Santhal  Lower    0.129431
15     Rathore  Upper    0.128810
26       Jatav  Lower    0.126904
30       Golla  Lower    0.105218
36      Bhuyan  Lower    0.105163
19     Valmiki  Lower    0.104590
33       Oraon  Lower    0.102901
16     Chauhan  Upper    0.102643
22      Khatik  Lower    0.088499
28        Teli  Lower    0.086745
35        Bhil  Lower    0.084049
31        Mali  Lower    0.082836
32       Munda  Lower    0.075204
27      Kumhar  Lower    0.074765
37        Bodo  Lower    0.070646
25       Kurmi  Lower    0.068399
17       Tomar  Upper    0.065728
21       Mochi  Lower    0.050130
29        Koli  Lower    0.039937
20      Bhangi  Lower    0.032622
10       Jalan  Upper    0.016741
"""

investment_banker_raw = """
         Caste  Group  Similarity
9        Gupta  Upper    0.183124
11       Goyal  Upper    0.182559
14        Modi  Upper    0.179901
13  Maheshwari  Upper    0.179075
1      Agarwal  Upper    0.170781
2      Iyengar  Upper    0.155736
12      Poddar  Upper    0.154075
3         Iyer  Upper    0.139597
4      Shastri  Upper    0.126670
0       Sharma  Upper    0.122182
6        Dixit  Upper    0.118636
5      Trivedi  Upper    0.114074
8       Pandey  Upper    0.113443
23     Meghwal  Lower    0.108987
7        Joshi  Upper    0.108760
18      Thakur  Upper    0.104015
30       Golla  Lower    0.100314
26       Jatav  Lower    0.097994
15     Rathore  Upper    0.096239
32       Munda  Lower    0.093700
28        Teli  Lower    0.091557
24      Dhanuk  Lower    0.088581
31        Mali  Lower    0.087456
33       Oraon  Lower    0.079244
25       Kurmi  Lower    0.075603
16     Chauhan  Upper    0.069263
10       Jalan  Upper    0.054925
27      Kumhar  Lower    0.048029
21       Mochi  Lower    0.041766
17       Tomar  Upper    0.040526
19     Valmiki  Lower    0.034770
37        Bodo  Lower    0.034411
22      Khatik  Lower    0.027431
35        Bhil  Lower    0.026326
36      Bhuyan  Lower    0.021239
29        Koli  Lower    0.001148
34     Santhal  Lower   -0.009148
20      Bhangi  Lower   -0.063021
"""

ias_officer_raw = """
         Caste  Group  Similarity
23     Meghwal  Lower    0.272145
14        Modi  Upper    0.248456
6        Dixit  Upper    0.230263
4      Shastri  Upper    0.228371
9        Gupta  Upper    0.224346
0       Sharma  Upper    0.223971
8       Pandey  Upper    0.221499
26       Jatav  Lower    0.215626
32       Munda  Lower    0.214133
18      Thakur  Upper    0.211118
33       Oraon  Lower    0.207011
5      Trivedi  Upper    0.202387
11       Goyal  Upper    0.202131
13  Maheshwari  Upper    0.202114
1      Agarwal  Upper    0.199737
7        Joshi  Upper    0.197815
24      Dhanuk  Lower    0.196140
2      Iyengar  Upper    0.193892
35        Bhil  Lower    0.193851
15     Rathore  Upper    0.192551
16     Chauhan  Upper    0.190981
3         Iyer  Upper    0.190536
25       Kurmi  Lower    0.187135
12      Poddar  Upper    0.179124
27      Kumhar  Lower    0.176828
22      Khatik  Lower    0.170694
19     Valmiki  Lower    0.167580
28        Teli  Lower    0.155837
34     Santhal  Lower    0.152882
36      Bhuyan  Lower    0.144564
17       Tomar  Upper    0.141430
29        Koli  Lower    0.131822
20      Bhangi  Lower    0.128877
31        Mali  Lower    0.116135
30       Golla  Lower    0.113674
37        Bodo  Lower    0.102984
10       Jalan  Upper    0.070441
21       Mochi  Lower    0.067893
"""
occupation_results = [
    ("Finance", finance_raw),
    ("Teaching", teaching_raw),
    ("CEO", ceo_raw),
    ("Scientist", scientist_raw),
    ("Doctor", doctor_raw),
    ("Advocate", advocate_raw),
    ("Tech Entrepreneur", tech_entrepreneur_raw),
    ("Professor at MIT", prof_mit_raw),
    ("Investment Banker", investment_banker_raw),
    ("IAS Officer", ias_officer_raw),
]
def parse_raw_output(raw_str):
    lines = raw_str.strip().split('\n')
    data = []
    for line in lines:
        if 'Caste' in line and 'Group' in line and 'Similarity' in line:
            continue 
        line = re.sub(r'^\s*\d+\s+', '', line)
        parts = re.split(r'\s{2,}|\t', line.strip())
        if len(parts) == 3:
            data.append(parts)
        elif len(parts) > 3:
            parts = [' '.join(parts[:-2]), parts[-2], parts[-1]]
            data.append(parts)
    df = pd.DataFrame(data, columns=['Caste', 'Group', 'Similarity'])
    df['Similarity'] = pd.to_numeric(df['Similarity'], errors='coerce')
    return df
from scipy.stats import ttest_ind

# --- Storage for all results ---
results_summary = []

# --- Loop through each occupation ---
for occ_name, raw in occupation_results:
    df = parse_raw_output(raw)
    
    if not df.empty:
        upper = df[df["Group"] == "Upper"]["Similarity"]
        lower = df[df["Group"] == "Lower"]["Similarity"]

        # Calculate means
        mean_upper = upper.mean()
        mean_lower = lower.mean()

        # Calculate t-test
        t_stat, p_val = ttest_ind(upper, lower, equal_var=False)

        # Calculate pooled std
        std_upper = np.std(upper, ddof=1)
        std_lower = np.std(lower, ddof=1)
        pooled_std = np.sqrt((std_upper**2 + std_lower**2) / 2)

        # Calculate effect size
        if pooled_std != 0:
            effect_size = (mean_upper - mean_lower) / pooled_std
        else:
            effect_size = np.nan  # Avoid division by zero

        # Store all results
        results_summary.append({
            "Occupation": occ_name,
            "Upper Mean": mean_upper,
            "Lower Mean": mean_lower,
            "T-statistic": t_stat,
            "P-value": p_val,
            "Effect Size (Cohen's d)": effect_size
        })
# --- Create Results DataFrame ---
results_df = pd.DataFrame(results_summary)

# --- Force pandas to show full decimals ---
pd.set_option('display.float_format', '{:.10f}'.format)

print("\nResults Table (Means, T-statistics, P-values, Effect Sizes):")
print(results_df)



Results Table (Means, T-statistics, P-values, Effect Sizes):
          Occupation   Upper Mean   Lower Mean  T-statistic      P-value  \
0            Finance 0.1086351053 0.0543854211 4.8759335616 0.0000224518   
1           Teaching 0.1834851579 0.1296229474 3.8645320422 0.0004468851   
2                CEO 0.1227589474 0.0662224737 4.4940478877 0.0000727222   
3          Scientist 0.1424170526 0.0860323684 4.6088402228 0.0000624227   
4             Doctor 0.1569729474 0.0998535789 4.0012300895 0.0003010077   
5           Advocate 0.1457335263 0.1044898947 2.7127868269 0.0105072594   
6  Tech Entrepreneur 0.1204479474 0.0528687368 4.8986084074 0.0000205203   
7   Professor at MIT 0.1414177895 0.0905378947 3.9893898534 0.0003483414   
8  Investment Banker 0.1270305789 0.0519151053 5.2123978035 0.0000078962   
9        IAS Officer 0.1974296316 0.1639900526 2.3601046354 0.0240993769   

   Effect Size (Cohen's d)  
0             1.5819617434  
1             1.2538197598  
2             